In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu pypdf pandas openpyxl

In [ ]:
import os
import re
import json
import torch
import faiss
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

PDF_PATH = "/content/Project_Management_Policy.pdf"
EXCEL_PATH = "/content/Project_Management_Data.xlsx"


# Load Embedding Model

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load LLM


model_name = "Qwen/Qwen2-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

print("LLM loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (13 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully.


In [ ]:

PDF_PATH = "/content/Project_Management_Policy.pdf"
EXCEL_PATH = "/content/Project_Management_Data.xlsx"

print(PDF_PATH)
print(EXCEL_PATH)


import pandas as pd
from pypdf import PdfReader



def load_pdf(path):

    reader = PdfReader(path)

    text = ""

    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"

    return text


def load_excel(path):

    df = pd.read_excel(path)

    rows = []

    for _, row in df.iterrows():

        row_text = " | ".join(
            f"{column}: {row[column]}"
            for column in df.columns
        )

        rows.append(row_text)

    return "\n".join(rows)



pdf_text = load_pdf(PDF_PATH)

excel_text = load_excel(EXCEL_PATH)



knowledge_base = (
    "PROJECT MANAGEMENT POLICY:\n"
    + pdf_text
    + "\n\nPROJECT MANAGEMENT DATA:\n"
    + excel_text
)


print("PDF characters:", len(pdf_text))
print("Excel characters:", len(excel_text))
print("Knowledge base ready!")

/content/Project_Management_Policy.pdf
/content/Project_Management_Data.xlsx


FileNotFoundError: [Errno 2] No such file or directory: '/content/Project_Management_Policy.pdf'

In [ ]:

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss


def create_chunks(text, chunk_size=500, overlap=100):

    words = text.split()

    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(words), step):

        chunk = " ".join(
            words[i:i + chunk_size]
        )

        if chunk.strip():
            chunks.append(chunk)

    return chunks


chunks = create_chunks(knowledge_base)

print("Number of chunks:", len(chunks))



embed_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded!")



embeddings = embed_model.encode(
    chunks,
    convert_to_numpy=True,
    show_progress_bar=True
)

embeddings = embeddings.astype("float32")

print("Embedding shape:", embeddings.shape)



faiss.normalize_L2(embeddings)


dimension = embeddings.shape[1]

vector_db = faiss.IndexFlatIP(dimension)

vector_db.add(embeddings)


print("FAISS vector database created!")
print("Vectors stored:", vector_db.ntotal)

In [ ]:


import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. LOAD LLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto"
)

print("LLM loaded successfully!")

# 2. GENERATE FUNCTION


def generate(prompt, max_new_tokens=300):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs.input_ids.shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


# 3. RETRIEVAL

def retrieve(question, k=5):

    query_embedding = embed_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    scores, indices = vector_db.search(
        query_embedding,
        k
    )

    results = []

    for i in indices[0]:

        if i < len(chunks):
            results.append(chunks[i])

    return results


# 4. PLANNER AGENT


def planner_agent(question):

    prompt = f"""
You are the Planner Agent.

Create a short plan for answering the user's question.

User Question:
{question}

Return 3 clear steps.
"""

    return generate(prompt, max_new_tokens=150)


# 5. RESEARCH AGENT

def research_agent(question):

    retrieved = retrieve(question)

    context = "\n\n".join(retrieved)

    prompt = f"""
You are the Research Agent.

Use ONLY the information in the context below.

Question:
{question}

Context:
{context}

Extract the most relevant facts needed to answer the question.
Do not invent information.
"""

    return generate(prompt, max_new_tokens=250)


# 6. WRITER AGENT


def writer_agent(question, research):

    prompt = f"""
You are the Writer Agent.

Write a structured report answering the user's question.

Question:
{question}

Research:
{research}

Use only the research provided.
"""

    return generate(prompt, max_new_tokens=300)

# 7. SUMMARY AGENT


def summary_agent(report):

    prompt = f"""
You are the Summary Agent.

Summarize the report in 3-5 concise bullet points.

Report:
{report}
"""

    return generate(prompt, max_new_tokens=150)


def run_system(question):

    print("=" * 60)
    print("MULTI-AGENT AI SYSTEM")
    print("=" * 60)

    print("\n[1] Planner Agent")
    plan = planner_agent(question)
    print(plan)

    print("\n[2] Research Agent")
    research = research_agent(question)
    print(research)

    print("\n[3] Writer Agent")
    report = writer_agent(question, research)
    print(report)

    print("\n[4] Summary Agent")
    summary = summary_agent(report)
    print(summary)

    final_report = f"""
# FINAL REPORT

## Question
{question}

## Report
{report}

## Summary
{summary}
"""

    return final_report

In [ ]:


question = "Which projects require the most attention and why?"

result = run_system(question)

print("\n\nFINAL REPORT:")
print(result)

In [ ]:

# TESTING & EVALUATION

test_questions = [

    "What are the most important project management risks?",

    "Which projects require the most attention?",

    "What does the company policy say about project management?",

    "What recommendations can be made based on the project data?"
]


print("=" * 70)
print("TESTING MULTI-AGENT SYSTEM")
print("=" * 70)

test_results = []

for i, question in enumerate(test_questions, 1):

    print(f"\nTEST {i}")
    print("-" * 70)
    print("Question:", question)

    try:

        result = run_system(question)

        success = (
            result is not None
            and len(result.strip()) > 100
        )

        test_results.append({
            "test": i,
            "question": question,
            "status": "PASS" if success else "FAIL"
        })

    except Exception as e:

        print("ERROR:", e)

        test_results.append({
            "test": i,
            "question": question,
            "status": "FAIL"
        })


print("\n" + "=" * 70)
print("TEST RESULTS")
print("=" * 70)

for result in test_results:

    print(
        f"Test {result['test']}: "
        f"{result['status']} | "
        f"{result['question']}"
    )

In [ ]:

!pip install -q gradio

In [ ]:
import gradio as gr


def gradio_agent(question):

    if not question or not question.strip():

        return "Please enter a question."

    try:

        result = run_system(question)

        return result

    except Exception as e:

        return f"System Error: {str(e)}"

# INTERFACE

with gr.Blocks() as demo:

    gr.Markdown(
        """
        # 🤖 Multi-Agent Project Management AI

        Ask a question about the project management
        data and company policy.

        The system uses:

        - 📄 PDF Policy
        - 📊 Excel Project Data
        - 🔎 RAG Retrieval
        - 🧠 Planner Agent
        - 🔬 Research Agent
        - ✍️ Writer Agent
        - 📝 Summary Agent
        """
    )

    question = gr.Textbox(
        label="Your Question",
        placeholder="Ask something about the projects...",
        lines=3
    )

    submit = gr.Button(
        "Generate Report"
    )

    output = gr.Markdown(
        label="Final Report"
    )

    submit.click(
        fn=gradio_agent,
        inputs=question,
        outputs=output
    )

    question.submit(
        fn=gradio_agent,
        inputs=question,
        outputs=output
    )

demo.launch(share=True)